<a href="https://colab.research.google.com/github/parisarahman/music-generation-unsupervised/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pretty_midi -q

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import pretty_midi
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/maestro/maestro-v3.0.0'
CSV_PATH     = os.path.join(DATASET_PATH, 'maestro-v3.0.0.csv')
SAVE_DIR     = '/content/drive/MyDrive/maestro'

metadata = pd.read_csv(CSV_PATH)

train_df = metadata[metadata['split'] == 'train']
val_df   = metadata[metadata['split'] == 'validation']
test_df  = metadata[metadata['split'] == 'test']

print(f'Train:      {len(train_df)}')
print(f'Validation: {len(val_df)}')
print(f'Test:       {len(test_df)}')

In [ ]:
FS          = 16
WINDOW_SIZE = 128
NOTE_DIM    = 88


D_MODEL    = 256
NHEAD      = 8
NUM_LAYERS = 4
DROPOUT    = 0.1


BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-4

In [ ]:
def midi_to_pianoroll(path, fs=FS):
    midi_data  = pretty_midi.PrettyMIDI(path)
    piano_roll = midi_data.get_piano_roll(fs=fs)
    piano_roll = piano_roll[21:109]
    piano_roll = (piano_roll > 0).astype(np.float32)
    piano_roll = piano_roll.T
    return piano_roll


def create_windows(piano_roll, window_size=WINDOW_SIZE):
    windows     = []
    total_steps = piano_roll.shape[0]
    for start in range(0, total_steps - window_size, window_size):
        window       = piano_roll[start:start + window_size]
        active_ratio = np.count_nonzero(window) / window.size
        if active_ratio > 0.02:
            windows.append(window)
    return windows


def build_and_save(df, save_path):
    """Process a split and save as .npy — skips if file already exists."""
    if os.path.exists(save_path):
        print(f'Already exists: {save_path}  →  skipping.')
        return

    all_windows = []
    for _, row in df.iterrows():
        path = os.path.join(DATASET_PATH, row['midi_filename'])
        try:
            pr      = midi_to_pianoroll(path)
            windows = create_windows(pr)
            all_windows.extend(windows)
        except Exception as e:
            print(f'Skipping {path}: {e}')

    arr = np.array(all_windows, dtype=np.float32)
    np.save(save_path, arr)
    print(f'Saved {arr.shape} → {save_path}')

In [ ]:
build_and_save(train_df.head(20), os.path.join(SAVE_DIR, 'train_pianoroll.npy'))
build_and_save(val_df.head(10),   os.path.join(SAVE_DIR, 'val_pianoroll.npy'))

In [ ]:
class PianoRollDataset(Dataset):
    def __init__(self, npy_path):
        self.data = np.load(npy_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32)


train_dataset = PianoRollDataset(os.path.join(SAVE_DIR, 'train_pianoroll.npy'))
val_dataset   = PianoRollDataset(os.path.join(SAVE_DIR, 'val_pianoroll.npy'))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train samples: {len(train_dataset)}')
print(f'Val   samples: {len(val_dataset)}')
print(f'Batch shape:   {next(iter(train_loader)).shape}')

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
class MusicTransformer(nn.Module):

    def __init__(self, note_dim=NOTE_DIM, d_model=D_MODEL,
                 nhead=NHEAD, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()

        self.input_proj  = nn.Linear(note_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out      = nn.Linear(d_model, note_dim)

    def forward(self, x):
      seq_len = x.size(1)
      causal_mask = nn.Transformer.generate_square_subsequent_mask(
        seq_len, device=x.device
      )
      x = self.input_proj(x)
      x = self.pos_encoder(x)
      x = self.transformer(x, mask=causal_mask, is_causal=True)
      x = self.fc_out(x)
      return x


model = MusicTransformer().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'transformer_best.pt')

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):


    model.train()
    running_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)

        src    = batch[:, :-1, :]
        target = batch[:, 1:,  :]

        optimizer.zero_grad()
        output = model(src)
        loss   = criterion(output, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    avg_train = running_loss / len(train_loader)
    train_losses.append(avg_train)


    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch  = batch.to(device)
            src    = batch[:, :-1, :]
            target = batch[:, 1:,  :]
            output = model(src)
            val_loss += criterion(output, target).item()

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'Train Loss: {avg_train:.4f}  Val Loss: {avg_val:.4f}')


    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'  ✔ Best model saved (val_loss={best_val_loss:.4f})')

print('Training complete!')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Music Transformer — Training Curve')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'transformer_loss.png'), dpi=150)
plt.show()

In [ ]:
def generate_music(model, seed_window, gen_steps=256, threshold=0.5):

    model.eval()
    generated = seed_window.tolist()

    with torch.no_grad():
        for _ in range(gen_steps):

            inp = torch.tensor(
                generated[-WINDOW_SIZE:], dtype=torch.float32
            ).unsqueeze(0).to(device)

            out    = model(inp)
            logits = out[0, -1, :]
            probs  = torch.sigmoid(logits).cpu().numpy()
            frame  = (probs > threshold).astype(np.float32)
            generated.append(frame.tolist())

    return np.array(generated, dtype=np.float32)



model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))


seed = val_dataset[0].numpy()

generated_roll = generate_music(model, seed, gen_steps=256)
print(f'Generated piano roll shape: {generated_roll.shape}')

In [ ]:
def pianoroll_to_midi(piano_roll, fs=FS, program=0, velocity=80):
    """
    piano_roll : (time_steps, 88) binary array
    returns    : pretty_midi.PrettyMIDI object
    """
    midi   = pretty_midi.PrettyMIDI()
    inst   = pretty_midi.Instrument(program=program)
    dt     = 1.0 / fs

    for note_idx in range(88):
        midi_pitch = note_idx + 21
        active     = False
        start_time = 0.0

        for t, frame in enumerate(piano_roll):
            if frame[note_idx] > 0 and not active:
                active     = True
                start_time = t * dt
            elif frame[note_idx] == 0 and active:
                active = False
                inst.notes.append(
                    pretty_midi.Note(
                        velocity=velocity,
                        pitch=midi_pitch,
                        start=start_time,
                        end=t * dt
                    )
                )

    midi.instruments.append(inst)
    return midi


output_midi = pianoroll_to_midi(generated_roll)
output_path = os.path.join(SAVE_DIR, 'transformer_generated.mid')
output_midi.write(output_path)
print(f'MIDI saved → {output_path}')

In [ ]:
plt.figure(figsize=(14, 4))
plt.imshow(
    generated_roll[:300].T,
    aspect='auto', origin='lower', cmap='Blues',
    interpolation='nearest'
)
plt.xlabel('Time Step')
plt.ylabel('MIDI Note (21–108)')
plt.title('Generated Piano Roll (first 300 frames)')
plt.colorbar(label='Note Active')
plt.tight_layout()
plt.show()

In [ ]:
def calculate_perplexity(model, data_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            batch  = batch.to(device)
            src    = batch[:, :-1, :]
            target = batch[:, 1:,  :]
            output = model(src)
            total_loss += criterion(output, target).item()
    avg_loss = total_loss / len(data_loader)
    return np.exp(avg_loss)

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
val_perplexity = calculate_perplexity(model, val_loader)
print(f'Validation Perplexity: {val_perplexity:.4f}')

In [ ]:
import os
os.makedirs(os.path.join(SAVE_DIR, 'transformer_samples'), exist_ok=True)

for i in range(10):
    seed = val_dataset[i].numpy()
    roll = generate_music(model, seed, gen_steps=256, threshold=0.3)
    midi = pianoroll_to_midi(roll)
    path = os.path.join(SAVE_DIR, f'transformer_samples/transformer_sample_{i}.mid')
    midi.write(path)
    print(f'Saved sample {i}')

In [ ]:
def pitch_histogram(midi_path):
    midi = pretty_midi.PrettyMIDI(midi_path)
    histogram = np.zeros(12)
    total_notes = 0
    for instrument in midi.instruments:
        for note in instrument.notes:
            histogram[note.pitch % 12] += 1
            total_notes += 1
    if total_notes > 0:
        histogram /= total_notes
    return histogram

def pitch_histogram_similarity(real_midi, generated_midi):
    return np.sum(np.abs(pitch_histogram(real_midi) - pitch_histogram(generated_midi)))

def rhythm_diversity(midi_path):
    midi = pretty_midi.PrettyMIDI(midi_path)
    durations = [round((n.end - n.start) / 0.05) * 0.05
                 for inst in midi.instruments for n in inst.notes]
    if not durations:
        return 0
    return len(set(durations)) / len(durations)

def repetition_ratio(midi_path, n=4):
    midi = pretty_midi.PrettyMIDI(midi_path)
    pitches = [n.pitch for inst in midi.instruments
               for n in sorted(inst.notes, key=lambda x: x.start)]
    patterns = [tuple(pitches[i:i+n]) for i in range(len(pitches) - n)]
    if not patterns:
        return 0
    counts = {}
    for p in patterns:
        counts[p] = counts.get(p, 0) + 1
    return sum(1 for v in counts.values() if v > 1) / len(patterns)

In [ ]:
transformer_midi = "/content/drive/MyDrive/maestro/transformer_generated.mid"
baseline_midi = "/content/drive/MyDrive/maestro/random_baseline.mid"

In [ ]:
p_transformer = pitch_histogram(transformer_midi)
p_baseline = pitch_histogram(baseline_midi)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.arange(12)

plt.figure(figsize=(10,5))

plt.bar(x - 0.2, p_baseline, width=0.4, label="Baseline (Markov/Random)")
plt.bar(x + 0.2, p_transformer, width=0.4, label="Transformer")

plt.xlabel("Pitch Class (0–11)")
plt.ylabel("Normalized Frequency")
plt.title("Transformer vs Baseline Pitch Histogram")
plt.legend()

plt.show()

In [ ]:
model = MusicTransformer().to(device)

In [ ]:
state_dict = torch.load("/content/drive/MyDrive/maestro/transformer_best.pt")

model.load_state_dict(state_dict)

In [ ]:
model.eval()

In [ ]:
model = MusicTransformer().to(device)

state_dict = torch.load("/content/drive/MyDrive/maestro/transformer_best.pt")

model.load_state_dict(state_dict)

model.eval()

In [ ]:
lstm_ph = 0.0882
lstm_rd = 0.0659
lstm_rr = 0.0000

vae_ph = 0.7368954014772457
vae_rd = 0.5
vae_rr = 0.0

#transformer Validation Perplexity
Validation_Perplexity= 1.1062



real_file = os.path.join(SAVE_DIR, 'markov.mid')


random_file = os.path.join(SAVE_DIR, 'random_baseline.mid')

if not os.path.exists(random_file):
    def random_music_generator(output_path, num_notes=200):
        midi = pretty_midi.PrettyMIDI()
        instrument = pretty_midi.Instrument(program=0)
        current_time = 0

        for i in range(num_notes):
            pitch = np.random.randint(21, 109)
            duration = np.random.choice([0.25, 0.5, 1.0])
            velocity = np.random.randint(60, 100)

            instrument.notes.append(
                pretty_midi.Note(
                    velocity=velocity,
                    pitch=pitch,
                    start=current_time,
                    end=current_time + duration
                )
            )
            current_time += duration

        midi.instruments.append(instrument)
        midi.write(output_path)

    random_music_generator(random_file)
    print("Random baseline generated!")



rand_ph = pitch_histogram_similarity(real_file, random_file)
rand_rd = rhythm_diversity(random_file)
rand_rr = repetition_ratio(random_file)


markov_file = os.path.join(SAVE_DIR, 'markov.mid')

mark_ph = pitch_histogram_similarity(real_file, markov_file)
mark_rd = rhythm_diversity(markov_file)
mark_rr = repetition_ratio(markov_file)



trans_ph = round(np.mean(ph), 4) if 'ph' in globals() else 0.0
trans_rd = round(np.mean(rd), 4) if 'rd' in globals() else 0.0
trans_rr = round(np.mean(rr), 4) if 'rr' in globals() else 0.0


results = pd.DataFrame({
    'Model':        ['Random', 'Markov', 'LSTM AE', 'VAE', 'Transformer'],

    'Pitch Sim ↓':  [
        round(rand_ph, 4),
        round(mark_ph, 4),
        lstm_ph,
        vae_ph,
        trans_ph
    ],

    'Rhythm Div ↑': [
        round(rand_rd, 4),
        round(mark_rd, 4),
        lstm_rd,
        vae_rd,
        trans_rd
    ],

    'Repet Ratio':  [
        round(rand_rr, 4),
        round(mark_rr, 4),
        lstm_rr,
        vae_rr,
        trans_rr
    ],

    'Perplexity':   [
        None,
        None,
        None,
        None,
        Validation_Perplexity
    ]
})


print("\n===== Final Comparison Table =====")
print(results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['Random', 'Markov', 'LSTM AE', 'VAE', 'Transformer']

pitch_sim   = [round(rand_ph,4), round(mark_ph,4), lstm_ph, vae_ph,  trans_ph]
rhythm_div  = [round(rand_rd,4), round(mark_rd,4), lstm_rd, vae_rd,  trans_rd]
repet_ratio = [round(rand_rr,4), round(mark_rr,4), lstm_rr, vae_rr,  trans_rr]

x     = np.arange(len(models))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Music Generation Model Comparison', fontsize=14, fontweight='bold')

# ── Plot 1: Pitch Sim, Rhythm Div, Repet Ratio ──
ax = axes[0]
b1 = ax.bar(x - width, pitch_sim,   width, label='Pitch Sim ↓',  color='#B5D4F4', edgecolor='#378ADD', linewidth=0.8)
b2 = ax.bar(x,         rhythm_div,  width, label='Rhythm Div ↑', color='#5DCAA5', edgecolor='#1D9E75', linewidth=0.8)
b3 = ax.bar(x + width, repet_ratio, width, label='Repet Ratio',  color='#F0997B', edgecolor='#D85A30', linewidth=0.8)

for bars in [b1, b2, b3]:
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=7.5, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.25)
ax.set_ylabel('Score')
ax.set_title('Evaluation Metrics by Model')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ── Plot 2: Perplexity (Transformer only) ──
ax2 = axes[1]
perp_models = ['Transformer']
perp_values = [Validation_Perplexity]

bars = ax2.bar(perp_models, perp_values, width=0.3,
               color='#CEC BF6', edgecolor='#534AB7', linewidth=0.8)
ax2.bar_label(bars, fmt='%.4f', padding=5, fontsize=10)
ax2.set_ylim(0, perp_values[0] * 1.5)
ax2.set_ylabel('Perplexity')
ax2.set_title('Transformer Validation Perplexity\n(lower = better)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved!")